# 3-Head Interview Question Classifier

DistilBERT с тремя головами:
- **Head 1**: Пунктуация/капитализация (12 классов) — предобучена, заморожена
- **Head 2**: Actionable vs non-actionable (бинарная) — основная задача
- **Head 3**: Query span detection (token-level) — где начинается вопрос

Датасет загружается автоматически из GitHub.

## 1. Установка зависимостей

In [ ]:
!pip install -q transformers datasets scikit-learn tqdm

## 2. Загрузка датасета из GitHub

In [ ]:
import urllib.request, json, os

DATASET_URL = (
    "https://raw.githubusercontent.com/"
    "russianoracle/interview-question-classifier/main/"
    "dataset/processed/dataset_normalized.jsonl"
)
DATASET_PATH = "dataset_normalized.jsonl"

if not os.path.exists(DATASET_PATH):
    print("Скачиваю датасет...")
    urllib.request.urlretrieve(DATASET_URL, DATASET_PATH)
    print("Готово.")
else:
    print("Датасет уже есть.")

samples = [json.loads(l) for l in open(DATASET_PATH) if l.strip()]

act  = sum(1 for s in samples if s['head2_label'] == 'actionable')
nact = len(samples) - act
print(f"Загружено: {len(samples)} записей  (actionable={act}, non_actionable={nact})")

## 3. Импорты и настройки

In [ ]:
from dataclasses import dataclass
from typing import Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "distilbert-base-multilingual-cased"

print(f"PyTorch {torch.__version__} | device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 4. Разбивка на train / val / test

In [ ]:
labels = [s['head2_label'] for s in samples]

train_idx, temp_idx = train_test_split(
    range(len(samples)), test_size=0.2, stratify=labels, random_state=42
)
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    stratify=[labels[i] for i in temp_idx],
    random_state=42,
)

train_samples = [samples[i] for i in train_idx]
val_samples   = [samples[i] for i in val_idx]
test_samples  = [samples[i] for i in test_idx]

for name, split in [("Train", train_samples), ("Val", val_samples), ("Test", test_samples)]:
    a = sum(1 for s in split if s['head2_label'] == 'actionable')
    print(f"{name}: {len(split)}  (act={a}, non_act={len(split)-a})")

## 5. Dataset & Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Vocab size: {len(tokenizer)}")


class QuestionDataset(Dataset):
    MAX_LEN   = 128
    LABEL2ID  = {"non_actionable": 0, "actionable": 1}
    ID2LABEL  = {0: "non_actionable", 1: "actionable"}

    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s   = self.samples[idx]
        enc = tokenizer(
            s['text'],
            max_length=self.MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        head2 = self.LABEL2ID[s['head2_label']]

        qs = s.get('head3_query_start') or 0
        head3 = np.zeros(self.MAX_LEN, dtype=np.int32)
        if qs > 0:
            for ti, wi in enumerate(enc.word_ids()):
                if wi is not None and wi >= qs:
                    head3[ti] = 1

        return {
            "input_ids":      enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "head2_label":    torch.tensor(head2, dtype=torch.long),
            "head3_labels":   torch.tensor(head3, dtype=torch.long),
        }


train_ds = QuestionDataset(train_samples)
val_ds   = QuestionDataset(val_samples)
test_ds  = QuestionDataset(test_samples)
print(f"Datasets: train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")

## 6. Модель

In [ ]:
class QuestionClassifier(nn.Module):
    def __init__(self, model_name: str):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        H = self.backbone.config.hidden_size

        # Head 2: бинарная классификация (CLS-токен)
        self.head2 = nn.Sequential(
            nn.Linear(H, 256), nn.Dropout(0.2), nn.ReLU(), nn.Linear(256, 2)
        )
        # Head 3: token-level span detection
        self.head3 = nn.Sequential(
            nn.Linear(H, 256), nn.Dropout(0.2), nn.ReLU(), nn.Linear(256, 2)
        )

    def forward(self, input_ids, attention_mask):
        out  = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        seq  = out.last_hidden_state          # (B, L, H)
        cls  = seq[:, 0, :]                   # (B, H)
        return self.head2(cls), self.head3(seq)


model = QuestionClassifier(MODEL_NAME).to(DEVICE)
total = sum(p.numel() for p in model.parameters())
train_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Параметры: {total:,} total / {train_p:,} trainable")

## 7. DataLoader-ы и оптимизатор

In [ ]:
BATCH_SIZE  = 32
LR          = 2e-5
NUM_EPOCHS  = 4
WARMUP      = 400

# Взвешенный сэмплер для балансировки классов при обучении
train_labels_num = [QuestionDataset.LABEL2ID[s['head2_label']] for s in train_samples]
cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=np.array(train_labels_num))
cw_tensor = torch.tensor(cw, dtype=torch.float32).to(DEVICE)
sample_w  = np.array([cw[y] for y in train_labels_num])
sampler   = WeightedRandomSampler(sample_w, len(sample_w), replacement=True)

nw = 0  # num_workers=0 безопаснее в Colab
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=nw)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,   num_workers=nw)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,   num_workers=nw)

optimizer = AdamW(model.parameters(), lr=LR)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP,
    num_training_steps=len(train_loader) * NUM_EPOCHS,
)

loss_h2 = nn.CrossEntropyLoss(weight=cw_tensor)
loss_h3 = nn.CrossEntropyLoss()

print(f"Batches: train={len(train_loader)}  val={len(val_loader)}  test={len(test_loader)}")
print(f"Class weights: {cw}")

## 8. Обучение

In [ ]:
def run_epoch(model, loader, optimizer=None, scheduler=None):
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        pbar = tqdm(loader, leave=False)
        for batch in pbar:
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            y2   = batch["head2_label"].to(DEVICE)
            y3   = batch["head3_labels"].to(DEVICE)

            logits2, logits3 = model(ids, mask)
            l2 = loss_h2(logits2, y2)
            l3 = loss_h3(logits3.view(-1, 2), y3.view(-1))
            loss = 0.7 * l2 + 0.3 * l3

            if training:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()

            total_loss += loss.item()
            preds = logits2.argmax(1)
            correct += (preds == y2).sum().item()
            total   += y2.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y2.cpu().numpy())
            pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{correct/total:.4f}")

    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    _, _, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="weighted", zero_division=0
    )
    return avg_loss, acc, f1


best_f1, patience, pat_cnt = 0.0, 2, 0

for epoch in range(NUM_EPOCHS):
    tr_loss, tr_acc, tr_f1 = run_epoch(model, train_loader, optimizer, scheduler)
    va_loss, va_acc, va_f1 = run_epoch(model, val_loader)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS}  "
          f"train loss={tr_loss:.4f} acc={tr_acc:.4f}  "
          f"val loss={va_loss:.4f} acc={va_acc:.4f} f1={va_f1:.4f}")

    if va_f1 > best_f1:
        best_f1, pat_cnt = va_f1, 0
        torch.save(model.state_dict(), "best_model.pth")
        print(f"  ✅ Сохранён лучший checkpoint (F1={va_f1:.4f})")
    else:
        pat_cnt += 1
        if pat_cnt >= patience:
            print("  Early stop.")
            break

model.load_state_dict(torch.load("best_model.pth"))
print("\nОбучение завершено. Загружен лучший checkpoint.")

## 9. Тест

In [ ]:
te_loss, te_acc, te_f1 = run_epoch(model, test_loader)
print(f"Test: loss={te_loss:.4f}  accuracy={te_acc:.4f}  F1={te_f1:.4f}")

## 10. Примеры инференса

In [ ]:
@torch.no_grad()
def predict(text: str):
    model.eval()
    enc = tokenizer(
        text, max_length=QuestionDataset.MAX_LEN,
        padding="max_length", truncation=True, return_tensors="pt"
    )
    logits2, _ = model(enc["input_ids"].to(DEVICE), enc["attention_mask"].to(DEVICE))
    probs = torch.softmax(logits2, dim=1)[0]
    label = QuestionDataset.ID2LABEL[probs.argmax().item()]
    return label, probs[1].item()


for text in [
    "Какие вопросы обычно задают на собеседовании?",
    "Расскажи о своём опыте работы с базами данных.",
    "Проект был завершён в срок.",
    "Как работает сборщик мусора в Java?",
    "Я занимался разработкой микросервисов на Go.",
]:
    label, conf = predict(text)
    mark = "❓" if label == "actionable" else "💬"
    print(f"{mark} [{conf:.2f}] {text}")

## 11. Сохранение весов

In [ ]:
torch.save(model.state_dict(), "question_classifier_weights.pth")
print("✅ question_classifier_weights.pth")

with open("model_config.json", "w") as f:
    json.dump({"model_name": MODEL_NAME, "max_len": QuestionDataset.MAX_LEN,
               "id2label": QuestionDataset.ID2LABEL}, f, indent=2)
print("✅ model_config.json")